# Neuron Skeletonization Service Notebook

Copyright (c) 2025 Open Brain Institute

+ Author(s): 
    - Michael W. Reimann < michael.reimann@openbraininstitute.org >
    - Marwan Abdellah < marwan.abdellah@openbraininstitute.org >
    - James Isbister < james.isbister@openbraininstitute.org >

Last modified: 10.2026

The skeletonization runs in this notebook's kernel, so it needs obi-one's `meshing` extra (`make install-meshing`). That installs Ultraliser from AWS CodeArtifact (see [docs/index.md](../../../../../docs/index.md)), which publishes it for Linux only.

# Imports

In [ ]:
from obi_one.core.info import Info
from obi_one.core.run_tasks import run_task_for_single_config
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.db_sdk import db_sdk

# All essential imports needed for the notebook
from IPython.display import display, HTML
from obi_auth import get_token
from obi_notebook import get_projects, get_entities, get_environment
from entitysdk.models import TaskActivity
from entitysdk.client import Client
from entitysdk.types import ActivityStatus, TaskActivityType
import os
from obi_one.scientific.from_id.em_cell_mesh_from_id import EMCellMeshFromID
from obi_one.scientific.tasks.skeletonization.config import SkeletonizationScanConfig

## Authentication and project selection
We authenticate with the OBI platform.
### Project selection
Select from the dropdown menu the project that the output (a morphology, by default with extracted spines) should be registered to. It will be available only in that project context. 

The widget lists all projects you have access to.

In [ ]:
# environment = get_environment.get_environment()
environment = "staging"
token = get_token(environment=environment)
os.environ["OBI_AUTHENTICATION_TOKEN"] = token
project_context = get_projects.get_projects(token, env=environment)

## Display EMCellMesh table

Input into the skeletonization process is a mesh describing the surface of a neuron. 

Below we will display a list of all surface meshes you have access to in the project context you selected above. This is a combination of publicly available entities open to all users and cell meshes you or your team have registered inside the project.
You can filter the list based on the EM dataset the meshes originate from. You can also search for specific neuron identifiers (typically called "pt_root_id").

Please select one mesh from the list below.

In [ ]:
mesh_ids = []
mesh_ids = get_entities.get_entities(
    'em-cell-mesh', token, [], env=environment, project_context=project_context, 
    multi_select=True, page_size=20, add_columns=["dense_reconstruction_cell_id"])

## Check the selection 

In [ ]:
if len(mesh_ids) == 0:
    display(HTML("<h3 style='color:#EDB95E'>⚠️ No mesh selected</h3>"))
    raise SystemExit()

Prepare

# Configure the skeletonization

Disable `remove_spines_from_skeleton` to keep small branches that are not spines in the skeleton, e.g. for retinal neurons; no spines are segmented then. Enable `repair_slicing_artifacts` if the mesh may have slicing artifacts from the EM reconstruction.

In [ ]:
cell_mesh_from_ids = [EMCellMeshFromID(id_str=mesh_id) for mesh_id in mesh_ids]

skeletonization_scan_conf = SkeletonizationScanConfig(
    info=Info(
        campaign_name="Skeletonization test campaign",
        campaign_description="Skeletonization test description",
    ),
    initialize=SkeletonizationScanConfig.Initialize(
        cell_mesh=cell_mesh_from_ids,
        neuron_voxel_size=0.1,
        spines_voxel_size=0.1,
        remove_spines_from_skeleton=True,
        repair_slicing_artifacts=False,
    ),
)

# Register the configurations

`GridScanGenerationTask` registers the campaign in the selected project, with one configuration per mesh.

In [ ]:
grid_scan_generation_task = GridScanGenerationTask(form=skeletonization_scan_conf, output_root='../../../../../../obi-output/skeletonization/grid_scan')

db_client = Client(environment=environment, token_manager=token, project_context=project_context)
grid_scan_generation_task.execute(db_client=db_client)

# Skeletonize each mesh

Each configuration runs in this kernel under an execution activity, created as the platform's launch service would create it. The task registers the reconstructed morphology in the selected project and links it to that activity.

In [ ]:
for single_config in grid_scan_generation_task.single_configs:
    activity = db_sdk.create_generic_activity(
        client=db_client,
        used=[single_config.single_entity],
        activity_type=TaskActivityType.skeletonization__execution,
        activity_status=ActivityStatus.running,
    )
    try:
        run_task_for_single_config(
            single_config, db_client=db_client, execution_activity_id=str(activity.id)
        )
    except Exception:
        db_sdk.finalize_activity(
            client=db_client, activity_id=activity.id, activity_type=TaskActivity,
            status=ActivityStatus.error,
        )
        raise
    db_sdk.finalize_activity(
        client=db_client, activity_id=activity.id, activity_type=TaskActivity,
        status=ActivityStatus.done,
    )
    activity = db_client.get_entity(entity_id=activity.id, entity_type=TaskActivity)
    morphology_ids = [str(entity.id) for entity in activity.generated or []]
    print(f"Mesh {single_config.initialize.cell_mesh.id_str}: morphology {morphology_ids}")